# 소비 고정 IBT → TOU 청구액 비교
두 노트북이 `micro_common_config.json`과 `micro_common.py`를 함께 읽습니다.
공통 표본·사용량·소득분위·기준 청구액은 동일합니다. 이 노트북은 실제 청구식을 비교하고,
CGE 연계 노트북은 모형 가격배율과 환류를 적용하므로 두 결과의 차이를 일반균형 효과로 해석하지 않습니다.


## 1. 패키지 준비

In [ ]:
# 최초 한 번 터미널에서 설치: python -m pip install pandas numpy openpyxl jupyterlab
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd


## 2. 공통 설정과 요금 상수
설정은 아래 JSON 한 곳에서 변경합니다. TOU 요금표와 계약전력 가정은 유지합니다.
기본 표본은 50~1000kWh입니다. 1000kWh 초과 TOU 청구식은 이번 단계에서 확장하지 않습니다.


In [ ]:
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Run_Master.gms').is_file())
ROOT = PROJECT/'Micro'
sys.path.insert(0,str(ROOT))
from micro_common import (BLOCKS,load_config,months_for,profile_tag,
    prepare_common_sample,export_common_sample,weighted_mean)
from micro_common import gross_factor as common_gross_factor
TOU_MILD = np.array([172.4,153.8,125.8])
TOU_HOT_COLD = np.array([220.5,184.7,138.7])


## 3. 입력 경로와 공통 분석 설정
`micro_common_config.json`에서 profile(other/summer/annual_flat), voltage,
사용량 범위, 세금·기금, 계약전력, 저소득 분위 범위를 변경합니다.
기본값은 기타계절 200/400kWh·기금 2.7%·아파트 고압·TOU 3kW입니다.
출력 폴더를 새로 사용하므로 이전 연평균 결과는 보존됩니다.


In [ ]:
CONFIG = load_config(ROOT/'micro_common_config.json')
INPUT_FILE = ROOT/'survey_data.xlsx'
months = months_for(CONFIG)
VOLTAGE = CONFIG['voltage']
CONTRACT_KW = CONFIG['contract_kw']
ADDON = CONFIG['climate_fuel_won_kwh']
def gross_factor(month):
    return common_gross_factor(month,CONFIG)
path = INPUT_FILE
out = ROOT/('output_jeju_common_'+profile_tag(CONFIG))
out.mkdir(parents=True,exist_ok=True)
display(pd.Series(CONFIG,name='setting').to_frame())


## 4. 자료 로드 및 사용량 역산

In [ ]:
d,h,sample_flow,income_summary,common_meta = prepare_common_sample(path,CONFIG)
common_dir = export_common_sample(ROOT,CONFIG,d,h,sample_flow,income_summary,common_meta)
cuts = np.array(common_meta['income_cutoffs_monthly_eq_won'])
display(sample_flow)
print('공통 분석 가구 수:',len(h))


## 5. 공통 표본과 균등화소득
월 균등화소득=가구 월소득/√가구원 수. 공통 분석 표본 안에서 가구 단위 가중 분위로 분류합니다.
동률은 같은 분위이며 분위별 가구 수는 정확히 20%가 아닐 수 있습니다.
930/2639/3997/5620/10404는 연간 만원으로 해석한 참고 평균이며 경곗값이 아닙니다.
소득 0은 표본에 유지하고 소득 대비 부담률에서만 제외합니다.


In [ ]:
display(income_summary)
print('월 균등화소득 분위 경계(원):',cuts)
print('공통 표본 저장 위치:',common_dir)


## 6. TOU 청구액과 CGE 가격충격 계산

In [ ]:
tou_rates=np.array([TOU_MILD if m in (3,4,5,9,10) else TOU_HOT_COLD for m in months])+ADDON
factors=np.array([gross_factor(m) for m in months])
fixed=4310.*CONTRACT_KW
d['bill_tou']=np.nan
mask=d.est_sample
q=d.loc[mask,'kwh_est'].to_numpy()
s=d.loc[mask,['s_'+b for b in BLOCKS]].to_numpy()
d.loc[mask,'bill_tou']=np.mean((fixed+q[:,None]*(s@tou_rates.T))*factors,axis=1)
d['delta_bill']=d.bill_tou-d.bill_ibt_reconstructed
d['delta_bill_pct']=100*d.delta_bill/d.bill_ibt_reconstructed.where(d.bill_ibt_reconstructed.gt(0))
d['delta_vs_reported']=d.bill_tou-d.a1
d['burden_change_pp']=100*d.delta_bill/d.income_won.where(d.income_won.gt(0))
use=d.loc[mask]
if use.empty: raise ValueError('No eligible records')
qsum=float(np.dot(use.weight,use.kwh_est))
# Both anchors are quantity-weighted aggregate ratios, not mean(bill/kWh).
anchor_energy=float(np.dot(use.weight,use.ibt_variable_net))/qsum
anchor_total=float(np.dot(use.weight,use.bill_ibt_reconstructed))/qsum
theta=(s*q[:,None]*use.weight.to_numpy()[:,None]).sum(axis=0)/qsum
# Uniform per-kWh allocation of aggregate fixed receipts, for accounting only.
total_prices=(tou_rates*factors[:,None]).mean(axis=0)+fixed*factors.mean()*use.weight.sum()/qsum
prices=tou_rates.mean(axis=0)
table=pd.DataFrame({'block':BLOCKS,'theta_kwh':theta,'tou_variable_net':prices,
    'bau_variable_anchor':anchor_energy,'wedge_energy_only':prices/anchor_energy-1,
    'tou_total_equivalent':total_prices,'bau_total_anchor':anchor_total,
    'wedge_bill_equivalent':total_prices/anchor_total-1})


## 7. 결과 저장 및 확인

In [ ]:
table.to_csv(out/'cge_price_candidates.csv',index=False,encoding='utf-8-sig')
d.to_csv(out/'household_results.csv',index=False,encoding='utf-8-sig')
columns={'mean_kwh':'kwh_est','mean_bau':'bill_ibt_reconstructed','mean_tou':'bill_tou',
         'mean_delta':'delta_bill','mean_delta_pct':'delta_bill_pct','mean_burden_change_pp':'burden_change_pp'}
summary=[]
for q,g in use.groupby('income_quintile'):
    summary.append(dict(income_quintile=int(q),n=len(g),weighted_n=g.weight.sum(),
                        **{name:weighted_mean(g,col) for name,col in columns.items()}))
summary=pd.DataFrame(summary)
summary.to_csv(out/'income_summary.csv',index=False,encoding='utf-8-sig')
meta=dict(common_meta,months=months,voltage_assumption=VOLTAGE,contract_kw_assumed=CONTRACT_KW,
    n_sample=len(use),anchor_energy_net=anchor_energy,anchor_total_gross=anchor_total,
    mean_bill_change_won=weighted_mean(use,'delta_bill'),
    aggregate_bill_change_pct=100*np.dot(use.weight,use.delta_bill)/np.dot(use.weight,use.bill_ibt_reconstructed),
    note='Fixed-kWh fixed-share direct tariff comparison; GAMS inputs are NOT changed.')
(out/'assumptions_and_summary.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
lines=['* ENERGY-ONLY CANDIDATE: not full bill effect; fixed charges excluded.',
       '* Common sample / equivalised income; candidate only. Existing GAMS inputs unchanged.',
       f"bau_unit_price('01_KOR') = {anchor_energy:.10f};"]
for b,price in zip(BLOCKS,prices):lines.append(f"tou_tariff('{b}','01_KOR') = {price:.10f};")
(out/'TOU_price_candidate.gms').write_text('\n'.join(lines)+'\n',encoding='utf-8')
display(table.round(6))
display(summary)
print('공통 표본 직접청구식 비교: 월평균 변화(원)',meta['mean_bill_change_won'])
print('저장 위치:',out)


## 해석과 범위
- 관측 A1을 설명하는 IBT 사용량을 역산한 뒤 동일 사용량·시간대 비중으로 TOU를 계산합니다.
- 아파트 고압은 계약 관측이 아닌 가정입니다. 평일 비중을 월 전체에 적용합니다.
- 기타계절과 여름 역산은 월평균 A1에 대한 조건부 시나리오입니다. 실제 계절별 수요 관측이 아닙니다.
- 기본요금 간극과 50~1000kWh 밖의 가구는 공통 표본에서 제외합니다.
- A8·소득·가구원 수가 유효한 표본을 사용해 U/I/C/IC 비교가 같은 가구에 적용되도록 합니다.
- 요금계수는 기존 노트북을 유지했으며 최신 요금 검증·변경은 이번 작업 범위가 아닙니다.
- 생성된 TOU_price_candidate.gms는 검토용 후보입니다. GAMS의 기존 연평균 요금 입력은 자동 변경하지 않습니다.
- 이 노트북과 CGE 연계 결과는 표본·기준값은 같지만 가격 적용 방식과 고정요금 처리가 다릅니다.
  결과 차이를 행동반응이나 일반균형 효과로 분해한 것으로 해석하지 않습니다.
